# Graph Representations for Proteins

This notebook covers how to represent proteins as graphs for GNN-based models.

**Learning Objectives:**
- Build k-NN and contact graphs from protein structures
- Define node and edge features
- Create PyTorch Geometric data objects

In [ ]:
# !pip install biotite torch torch-geometric -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from biotite.database import rcsb
import biotite.structure as struc
import biotite.structure.io.pdb as pdb

np.random.seed(42)

## 1. Load Structure

In [ ]:
# Load ubiquitin
pdb_path = rcsb.fetch('1UBQ', 'pdb', target_path='.')
pdb_file = pdb.PDBFile.read(pdb_path)
structure = pdb_file.get_structure(model=1)
protein = structure[struc.filter_amino_acids(structure)]

# Extract CA coordinates and sequence
ca_mask = protein.atom_name == 'CA'
ca_coords = protein.coord[ca_mask]

AA_3TO1 = {'ALA':'A','CYS':'C','ASP':'D','GLU':'E','PHE':'F',
           'GLY':'G','HIS':'H','ILE':'I','LYS':'K','LEU':'L',
           'MET':'M','ASN':'N','PRO':'P','GLN':'Q','ARG':'R',
           'SER':'S','THR':'T','VAL':'V','TRP':'W','TYR':'Y'}
sequence = ''.join(AA_3TO1.get(n, 'X') for n in protein.res_name[ca_mask])

print(f"Protein: {len(sequence)} residues")
print(f"Sequence: {sequence}")

## 2. K-Nearest Neighbor Graph

In [ ]:
def compute_distance_matrix(coords):
    diff = coords[:, None, :] - coords[None, :, :]
    return np.sqrt(np.sum(diff ** 2, axis=-1))

def build_knn_graph(coords, k=10):
    """
    Build k-nearest neighbor graph.
    
    Returns:
        edge_index: (2, E) source and target nodes
        edge_attr: (E,) distances
    """
    n = len(coords)
    dist_matrix = compute_distance_matrix(coords)
    
    edges_src = []
    edges_dst = []
    distances = []
    
    for i in range(n):
        dists = dist_matrix[i].copy()
        dists[i] = np.inf  # Exclude self
        
        neighbors = np.argsort(dists)[:k]
        for j in neighbors:
            edges_src.append(i)
            edges_dst.append(j)
            distances.append(dist_matrix[i, j])
    
    edge_index = np.array([edges_src, edges_dst])
    edge_attr = np.array(distances)
    
    return edge_index, edge_attr

# Build 10-NN graph
edge_index_knn, edge_attr_knn = build_knn_graph(ca_coords, k=10)
print(f"k-NN graph: {edge_index_knn.shape[1]} edges")
print(f"Edge distance range: {edge_attr_knn.min():.1f} - {edge_attr_knn.max():.1f} Å")

In [ ]:
# Analyze edge distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Edge length distribution
axes[0].hist(edge_attr_knn, bins=30, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Edge Distance (Å)')
axes[0].set_ylabel('Count')
axes[0].set_title('Edge Length Distribution (10-NN)')

# Sequence separation of edges
seq_sep = np.abs(edge_index_knn[0] - edge_index_knn[1])
axes[1].hist(seq_sep, bins=range(0, max(seq_sep)+2), edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Sequence Separation |i-j|')
axes[1].set_ylabel('Count')
axes[1].set_title('Edge Sequence Separation (10-NN)')

plt.tight_layout()
plt.show()

## 3. Contact Graph

In [ ]:
def build_contact_graph(coords, threshold=8.0):
    """
    Build contact graph based on distance threshold.
    
    Returns:
        edge_index: (2, E) source and target nodes
        edge_attr: (E,) distances
    """
    dist_matrix = compute_distance_matrix(coords)
    
    # Find contacts (excluding self)
    contacts = (dist_matrix < threshold) & (dist_matrix > 0)
    src, dst = np.where(contacts)
    
    edge_index = np.array([src, dst])
    edge_attr = dist_matrix[src, dst]
    
    return edge_index, edge_attr

# Build contact graph (8Å threshold)
edge_index_contact, edge_attr_contact = build_contact_graph(ca_coords, threshold=8.0)
print(f"Contact graph: {edge_index_contact.shape[1]} edges")

In [ ]:
# Compare graph sparsity for different thresholds
thresholds = [6, 8, 10, 12, 15, 20]
n_edges = []

for t in thresholds:
    ei, _ = build_contact_graph(ca_coords, threshold=t)
    n_edges.append(ei.shape[1] // 2)  # Divide by 2 for undirected

plt.figure(figsize=(8, 5))
plt.plot(thresholds, n_edges, 'bo-', markersize=10)
plt.xlabel('Distance Threshold (Å)')
plt.ylabel('Number of Edges (undirected)')
plt.title('Graph Density vs. Contact Threshold')
plt.grid(True, alpha=0.3)

# Max possible edges
n = len(ca_coords)
max_edges = n * (n - 1) // 2
plt.axhline(y=max_edges, color='r', linestyle='--', label=f'Max edges ({max_edges})')
plt.legend()
plt.show()

## 4. Node Features

In [ ]:
# One-hot amino acid encoding
AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

def one_hot_encode(sequence):
    encoding = np.zeros((len(sequence), 20), dtype=np.float32)
    for i, aa in enumerate(sequence):
        if aa in AA_TO_IDX:
            encoding[i, AA_TO_IDX[aa]] = 1.0
    return encoding

node_features = one_hot_encode(sequence)
print(f"Node features shape: {node_features.shape}")

In [ ]:
# Add positional encoding (sinusoidal)
def positional_encoding(n_positions, d_model=16):
    """Sinusoidal positional encoding."""
    position = np.arange(n_positions)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    
    pe = np.zeros((n_positions, d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    
    return pe.astype(np.float32)

pos_enc = positional_encoding(len(sequence), d_model=16)

# Combine features
node_features_full = np.concatenate([node_features, pos_enc], axis=1)
print(f"Full node features: {node_features_full.shape}")

## 5. Edge Features

In [ ]:
def compute_edge_features(edge_index, coords, sequence):
    """
    Compute rich edge features:
    - Distance (1D)
    - Sequence separation (1D)
    - Direction vector (3D)
    """
    src = edge_index[0]
    dst = edge_index[1]
    
    # Distance
    direction = coords[dst] - coords[src]
    distance = np.linalg.norm(direction, axis=1, keepdims=True)
    
    # Normalize direction
    direction_norm = direction / (distance + 1e-6)
    
    # Sequence separation (normalized)
    seq_sep = np.abs(src - dst)[:, np.newaxis] / len(sequence)
    
    # Combine
    edge_features = np.concatenate([
        distance / 10.0,  # Normalize by typical max distance
        seq_sep,
        direction_norm
    ], axis=1)
    
    return edge_features.astype(np.float32)

edge_features = compute_edge_features(edge_index_knn, ca_coords, sequence)
print(f"Edge features shape: {edge_features.shape}")
print(f"Features: [distance, seq_sep, dir_x, dir_y, dir_z]")

## 6. PyTorch Geometric Data Object

In [ ]:
try:
    from torch_geometric.data import Data, Batch
    
    def protein_to_pyg(coords, sequence, k=10):
        """
        Convert protein to PyTorch Geometric Data object.
        """
        # Build graph
        edge_index, _ = build_knn_graph(coords, k=k)
        
        # Node features (one-hot)
        x = torch.tensor(one_hot_encode(sequence), dtype=torch.float)
        
        # Edge features
        edge_attr = torch.tensor(
            compute_edge_features(edge_index, coords, sequence),
            dtype=torch.float
        )
        
        # Edge index
        edge_index = torch.tensor(edge_index, dtype=torch.long)
        
        # Coordinates
        pos = torch.tensor(coords, dtype=torch.float)
        
        return Data(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr,
            pos=pos,
            sequence=sequence
        )
    
    # Create data object
    data = protein_to_pyg(ca_coords, sequence, k=10)
    
    print("PyTorch Geometric Data Object:")
    print(f"  x (node features): {data.x.shape}")
    print(f"  edge_index: {data.edge_index.shape}")
    print(f"  edge_attr: {data.edge_attr.shape}")
    print(f"  pos (coordinates): {data.pos.shape}")
    print(f"  num_nodes: {data.num_nodes}")
    print(f"  num_edges: {data.num_edges}")
    
except ImportError:
    print("torch_geometric not installed. Run: pip install torch-geometric")

In [ ]:
# Batching multiple proteins
try:
    # Create a few "proteins" (same one, but demonstrates batching)
    proteins = [protein_to_pyg(ca_coords, sequence, k=10) for _ in range(4)]
    batch = Batch.from_data_list(proteins)
    
    print("\nBatched Data:")
    print(f"  x: {batch.x.shape}")
    print(f"  edge_index: {batch.edge_index.shape}")
    print(f"  batch tensor: {batch.batch.shape}")
    print(f"  Unique batch IDs: {torch.unique(batch.batch).tolist()}")
    
except NameError:
    print("Skipping batching demo (torch_geometric not available)")

## 7. Visualizing the Graph

In [ ]:
# Visualize graph as adjacency matrix
def edge_index_to_adjacency(edge_index, n_nodes):
    adj = np.zeros((n_nodes, n_nodes))
    adj[edge_index[0], edge_index[1]] = 1
    return adj

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# k-NN adjacency
adj_knn = edge_index_to_adjacency(edge_index_knn, len(sequence))
axes[0].imshow(adj_knn, cmap='Blues', origin='lower')
axes[0].set_xlabel('Residue')
axes[0].set_ylabel('Residue')
axes[0].set_title('10-NN Graph Adjacency')

# Contact adjacency
adj_contact = edge_index_to_adjacency(edge_index_contact, len(sequence))
axes[1].imshow(adj_contact, cmap='Blues', origin='lower')
axes[1].set_xlabel('Residue')
axes[1].set_ylabel('Residue')
axes[1].set_title('Contact Graph (8Å) Adjacency')

plt.tight_layout()
plt.show()

## Summary

| Graph Type | Edges | Properties |
|------------|-------|------------|
| k-NN | Fixed per node | Good for message passing |
| Contact | Variable | Captures structural contacts |
| Sequential | Backbone chain | Captures sequence order |

**PyTorch Geometric Data object contains:**
- `x`: Node features (L, F)
- `edge_index`: Edge connectivity (2, E)
- `edge_attr`: Edge features (E, D)
- `pos`: Node positions (L, 3)
- `batch`: Batch assignment for batched graphs